# VietHandOCR Part 1: Data Preparation & EDA

Welcome to **Part 1** of the VietHandOCR pipeline. 
- **Next Notebook**: [Part 2: Digital Image Processing (DIP) Pipeline](./02_Digital_Image_Processing.ipynb)

## Introduction
This notebook handles extracting the UIT-HWDB dataset, optimizing memory (downcasting), and performing a strict writer-independent split. We save the intermediate results to `.parquet` files for the next notebook.


In [ ]:
import os, gc, zipfile, random
import numpy as np
import pandas as pd

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)
print("Seeded everything successfully!")


In [ ]:
def reduce_mem_usage(df):
    '''Iterates through columns and modifies data types to reduce memory usage.'''
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and not pd.api.types.is_categorical_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max: df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                else: df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max: df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
                else: df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory decreased from {start_mem:.2f}MB to {end_mem:.2f}MB')
    return df


In [ ]:
def writer_independent_split(metadata_df, val_size=0.1):
    '''Splits data ensuring writers in validation set are not in training set.'''
    if 'writer_id' not in metadata_df.columns:
        from sklearn.model_selection import train_test_split
        return train_test_split(metadata_df, test_size=val_size, random_state=42)
        
    writers = metadata_df['writer_id'].unique()
    np.random.shuffle(writers)
    split_idx = int(len(writers) * (1 - val_size))
    train_writers, val_writers = writers[:split_idx], writers[split_idx:]
    
    train_df = metadata_df[metadata_df['writer_id'].isin(train_writers)].copy()
    val_df = metadata_df[metadata_df['writer_id'].isin(val_writers)].copy()
    return train_df, val_df

# Example:
# metadata = pd.read_csv('metadata.csv')
# metadata = reduce_mem_usage(metadata)
# train_df, val_df = writer_independent_split(metadata)
# train_df.to_parquet('train_metadata.parquet')
# val_df.to_parquet('val_metadata.parquet')
# del metadata, train_df, val_df
# gc.collect()
